# Understanding 2D Linear Dynamical Systems

This notebook explores the behavior of 2D linear dynamical systems. 

A 2D linear system is described by a set of two coupled first-order ordinary differential equations (ODEs):

$$ 
\frac{dx}{dt} = ax + by \\

\frac{dy}{dt} = cx + dy 
$$

We can write this in a more compact matrix form:

$$ 
\frac{d\mathbf{x}}{dt} = A \mathbf{x} 
$$

where $\mathbf{x} = \begin{pmatrix} x \\ y \end{pmatrix}$ and $A = \begin{pmatrix} a & b \\ c & d \end{pmatrix}$.

The point $(0,0)$ is always a fixed point (or equilibrium point) for these systems, because if $\mathbf{x} = \mathbf{0}$, then $\frac{d\mathbf{x}}{dt} = \mathbf{0}$. The stability and nature of this fixed point are entirely determined by the **eigenvalues** of the matrix $A$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

plt.xkcd()

## The Magic of Eigenvalues

The general solution to $\frac{d\mathbf{x}}{dt} = A \mathbf{x}$ is given by:

$$ \mathbf{x}(t) = c_1 \mathbf{v}_1 e^{\lambda_1 t} + c_2 \mathbf{v}_2 e^{\lambda_2 t} $$

where $\lambda_1, \lambda_2$ are the eigenvalues of $A$, and $\mathbf{v}_1, \mathbf{v}_2$ are the corresponding eigenvectors. The constants $c_1, c_2$ are determined by the initial conditions.

- If an eigenvalue $\lambda$ has a **negative real part**, the term $e^{\lambda t}$ decays to zero as $t \to \infty$.
- If an eigenvalue $\lambda$ has a **positive real part**, the term $e^{\lambda t}$ grows to infinity as $t \to \infty$.
- If an eigenvalue $\lambda$ has a **zero real part** (i.e., it's purely imaginary), the term $e^{\lambda t}$ oscillates without decay or growth.

We will now create a helper function to visualize the system's behavior by plotting its **phase portrait**. A phase portrait shows the vector field of the system and several example trajectories.

In [ ]:
def plot_phase_portrait(A, title="Phase Portrait", plot_trajectories=True):
    """
    Plots the phase portrait for a 2D linear system dx/dt = Ax.
    Also plots the eigenvectors for real eigenvalues.

    Args:
        A (numpy.ndarray): The 2x2 matrix defining the system.
        title (str): The title for the plot.
    """

    # Define the system of ODEs
    def system(X, t, A):
        return A @ X

    # Create a grid for the vector field
    x_max, y_max = 3.0, 3.0
    x = np.linspace(-x_max, x_max, 20)
    y = np.linspace(-y_max, y_max, 20)
    X1, Y1 = np.meshgrid(x, y)

    # Calculate the vector field at each grid point (vectorized for efficiency)
    X_grid = np.vstack([X1.ravel(), Y1.ravel()])
    dXdt_grid = A @ X_grid
    u, v = dXdt_grid[0, :].reshape(X1.shape), dXdt_grid[1, :].reshape(Y1.shape)

    # Plot the vector field
    plt.figure(figsize=(8, 7))
    plt.quiver(X1, Y1, u, v, color="gray", alpha=0.8, headlength=4, headaxislength=4)

    # Plot trajectories from different initial conditions
    t = np.linspace(0, 10, 1000)
    if plot_trajectories:
        initial_conditions = [
            [0.1, 0.1],
            [-0.1, 0.1],
            [0.1, -0.1],
            [-0.1, -0.1],
            [1, 1],
            [-1, 1],
            [1, -1],
            [-1, -1],
            [2, 0],
            [-2, 0],
            [0, 2],
            [0, -2],
        ]
    else:
        initial_conditions = []
    # Calculate and display eigenvalues and eigenvectors
    eigenvalues, eigenvectors = np.linalg.eig(A)
    eig_str = f"Eigenvalues: {eigenvalues[0]:.2f}, {eigenvalues[1]:.2f}"

    # Plot eigenvectors if they are real
    if not np.iscomplexobj(eigenvectors):
        for i in range(eigenvectors.shape[1]):
            v = eigenvectors[:, i].real
            # Plot a line extending in both directions from the origin
            plt.plot(
                [-v[0] * x_max * 2, v[0] * x_max * 2],
                [-v[1] * y_max * 2, v[1] * y_max * 2],
                "-",
                lw=5,
                label=f"Eigenvector for $\lambda$={eigenvalues[i]:.2f}",
                color="slateblue" if i == 0 else "midnightblue",
            )
        plt.legend()
    for x0 in initial_conditions:
        sol = odeint(system, x0, t, args=(A,))
        plt.plot(sol[:, 0], sol[:, 1], "-", color="darkorchid", lw=1, alpha=0.6)
        plt.plot(x0[0], x0[1], "o", color="darkorchid", markersize=4)  # Start point

    # Plot settings
    plt.xlabel("x")
    plt.ylabel("y")
    plt.xlim([-x_max, x_max])
    plt.ylim([-y_max, y_max])
    plt.title(f"{title}\n{eig_str}")
    plt.gca().set_aspect("equal", adjustable="box")
    plt.show()

## Case 1: Real and Distinct Eigenvalues ($\lambda_1 \neq \lambda_2$)

### 1a. Stable Node (Sink)
- **Condition**: $\lambda_1 < \lambda_2 < 0$ (both eigenvalues are negative).
- **Behavior**: All trajectories are pulled towards the fixed point at the origin. The origin is a stable equilibrium.

In [ ]:
# A matrix with two distinct negative real eigenvalues
A_stable_node = np.array([[-2, 1], [1, -2]])
plot_phase_portrait(A_stable_node, title="Stable Node (Sink).", plot_trajectories=True)

### Visualizing the Eigenvector Basis

For any system with real, distinct eigenvalues, the corresponding eigenvectors ($\mathbf{v}_1, \mathbf{v}_2$) are linearly independent and form a **basis** for the plane. This means that any initial condition $\mathbf{x}_0$ can be uniquely expressed as a linear combination of these eigenvectors:

$$ \mathbf{x}_0 = c_1 \mathbf{v}_1 + c_2 \mathbf{v}_2 $$

The plot below illustrates this decomposition. The vector $\mathbf{x}_0$ is the sum of its components along the eigenvector directions. This is powerful because we know how the system evolves along these special directions (purely exponential growth or decay). The evolution of $\mathbf{x}(t)$ is then just the sum of these evolving components:

$$ \mathbf{x}(t) = c_1 \mathbf{v}_1 e^{\lambda_1 t} + c_2 \mathbf{v}_2 e^{\lambda_2 t} $$

In [ ]:
# Eigenvector decomposition illustration
# Let's use the stable node case as an example
A = np.array([[-2, 1], [1, -2]])
x0 = np.array([3, 4])

# Get eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(A)
v1 = eigenvectors[:, 0]
v2 = eigenvectors[:, 1]

# The eigenvectors form a basis. We can find the coordinates of x0 in this basis.
# Since A is symmetric, the eigenvectors are orthogonal and normalized.
# We can project x0 onto v1 and v2 to find the coefficients c1 and c2.
c1 = np.dot(x0, v1)
c2 = np.dot(x0, v2)

# Start plotting
with plt.xkcd():
    plt.figure(figsize=(8, 7))
    ax = plt.gca()

    # Plot the components c1*v1 and c2*v2
    # Plot c1*v1 from the origin
    ax.arrow(
        0,
        0,
        c1 * v1[0],
        c1 * v1[1],
        head_width=0.1,
        head_length=0.1,
        fc="salmon",
        ec="salmon",
        linestyle="--",
        label=f"$c_1\mathbf{{v}}_1$",
    )
    # Plot c2*v2 starting from the end of c1*v1
    ax.arrow(
        c1 * v1[0],
        c1 * v1[1],
        c2 * v2[0],
        c2 * v2[1],
        head_width=0.1,
        head_length=0.1,
        fc="limegreen",
        ec="limegreen",
        linestyle="--",
        label=f"$c_2\mathbf{{v}}_2$",
    )

    # Plot the initial condition vector x0
    # ax.arrow(0, 0, x0[0], x0[1], head_width=0.1, head_length=0.1, fc='b', ec='b', lw=2, label='$\mathbf{x}_0$ (Initial Condition)')

    # Plot the eigenvectors (as basis vectors), scaled for visibility
    ax.arrow(
        0,
        0,
        2.5 * v1[0],
        2.5 * v1[1],
        head_width=0.08,
        head_length=0.08,
        fc="k",
        ec="k",
        lw=1,
        label="$\mathbf{v}_1$ (Eigenvector)",
    )
    ax.arrow(
        0,
        0,
        2.5 * v2[0],
        2.5 * v2[1],
        head_width=0.08,
        head_length=0.08,
        fc="slateblue",
        ec="slateblue",
        lw=1,
        label="$\mathbf{v}_2$ (Eigenvector)",
    )

    # Plot the point x0
    plt.plot(x0[0], x0[1], "bo", markersize=8)
    # plt.text(x0[0] + 0.1, x0[1], '$\mathbf{x}_0$', fontsize=16)

    # Settings
    plt.axhline(0, color="grey", linewidth=0.5)
    plt.axvline(0, color="grey", linewidth=0.5)
    plt.grid(True)
    plt.legend(fontsize=12)
    plt.title("Initial Condition as a Linear Combination of Eigenvectors")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.gca().set_aspect("equal", adjustable="box")
    plt.show()

### 1b. Unstable Node (Source)
- **Condition**: $\lambda_1 > \lambda_2 > 0$ (both eigenvalues are positive).
- **Behavior**: All trajectories are pushed away from the origin. The origin is an unstable equilibrium.

In [ ]:
# A matrix with two distinct positive real eigenvalues
A_unstable_node = np.array([[2, 1], [1, 2]])
plot_phase_portrait(
    A_unstable_node, title="Unstable Node (Source)", plot_trajectories=True
)

### 1c. Saddle Point
- **Condition**: $\lambda_1 < 0 < \lambda_2$ (eigenvalues have opposite signs).
- **Behavior**: Trajectories are attracted to the origin along one direction (the stable eigenvector) and repelled from it along another direction (the unstable eigenvector). The origin is an unstable equilibrium.

In [ ]:
# A matrix with one positive and one negative real eigenvalue
A_saddle = np.array([[2, 1], [-1, -1]])
plot_phase_portrait(A_saddle, title="Saddle Point", plot_trajectories=True)

## Case 2: Complex Conjugate Eigenvalues ($\lambda = \alpha \pm i\beta$)

When eigenvalues are complex, they always appear in a conjugate pair. The real part, $\alpha$, determines stability (inward/outward movement), while the imaginary part, $\beta$, determines the rotation.

### 2a. Stable Spiral (Spiral Sink)
- **Condition**: $\alpha < 0$ (real part is negative).
- **Behavior**: Trajectories spiral inwards towards the origin. The origin is a stable equilibrium.

In [ ]:
# A matrix with complex eigenvalues with a negative real part
A_stable_spiral = np.array([[-0.2, -1], [1, -0.2]])
plot_phase_portrait(A_stable_spiral, title="Stable Spiral")

### 2b. Unstable Spiral (Spiral Source)
- **Condition**: $\alpha > 0$ (real part is positive).
- **Behavior**: Trajectories spiral outwards, away from the origin. The origin is an unstable equilibrium.

In [ ]:
# A matrix with complex eigenvalues with a positive real part
A_unstable_spiral = np.array([[0.2, -1], [1, 0.2]])
plot_phase_portrait(A_unstable_spiral, title="Unstable Spiral")

### 2c. Center
- **Condition**: $\alpha = 0$ (real part is zero, eigenvalues are purely imaginary).
- **Behavior**: Trajectories form closed orbits (ellipses or circles) around the origin. The system is stable but not asymptotically stable, as trajectories do not approach the origin. This is a delicate, borderline case.

In [ ]:
# A matrix with purely imaginary eigenvalues
A_center = np.array([[0, -1], [1, 0]])
plot_phase_portrait(A_center, title="Center")

## Case 3: Repeated Eigenvalues ($\lambda_1 = \lambda_2 = \lambda$)

This is another borderline case. The behavior depends on whether the repeated eigenvalue has two linearly independent eigenvectors or only one.

### 3a. Star Node (Proper Node)
- **Condition**: $\lambda_1 = \lambda_2$ and there are two linearly independent eigenvectors. This only happens if the matrix $A$ is a multiple of the identity matrix, $A = \begin{pmatrix} \lambda & 0 \\ 0 & \lambda \end{pmatrix}$.
- **Behavior**: Trajectories are straight lines pointing directly towards (if $\lambda < 0$) or away from (if $\lambda > 0$) the origin.

In [ ]:
# A matrix for a stable star node
A_star_node = np.array([[-2, 0], [0, -2]])
plot_phase_portrait(A_star_node, title="Stable Star Node")

### 3b. Degenerate Node (Improper Node)
- **Condition**: $\lambda_1 = \lambda_2$ but there is only one linearly independent eigenvector.
- **Behavior**: Trajectories are curved. They approach the origin tangent to the single eigenvector direction (for a stable node).

In [ ]:
# A matrix for a stable degenerate node
A_degenerate_node = np.array([[-1, 1], [0, -1]])
plot_phase_portrait(A_degenerate_node, title="Stable Degenerate Node")

### Intuition: Why do Complex Eigenvalues cause Spirals? (Euler's Formula)

That's a fantastic question! The link between complex numbers and rotation is one of the most beautiful ideas in mathematics, and it's perfectly illustrated by these dynamical systems. The key is **Euler's Formula**.

Let's look at the general solution again, which involves terms like $e^{\lambda t}$. When the eigenvalues are a complex conjugate pair, $\lambda = \alpha \pm i\beta$, the solution is built from terms like:

$$ e^{\lambda t} = e^{(\alpha + i\beta)t} = e^{\alpha t} \cdot e^{i\beta t} $$

Now, let's apply Euler's Formula, which states: $e^{i\theta} = \cos(\theta) + i\sin(\theta)$.

Applying this to our term $e^{i\beta t}$:

$$ e^{\lambda t} = e^{\alpha t} (\cos(\beta t) + i\sin(\beta t)) $$

This single expression elegantly separates the two types of motion we see in the phase portrait:

1.  **Magnitude Change (Spiraling In/Out):** The $e^{\alpha t}$ term is a pure real exponential. The real part of the eigenvalue, $\alpha$, determines stability.
    *   If $\alpha < 0$, this term causes the solution to decay, pulling the trajectory **inward** towards the origin.
    *   If $\alpha > 0$, this term causes the solution to grow, pushing the trajectory **outward**.
    *   If $\alpha = 0$, this term is $e^0 = 1$, so the magnitude doesn't change over time.

2.  **Rotation (The "Spiral" part):** The $(\cos(\beta t) + i\sin(\beta t))$ term creates the rotation. The cosine and sine functions are inherently periodic and describe the coordinates of a point moving on a circle. The imaginary part of the eigenvalue, $\beta$, determines the speed of this rotation.

**Putting it together:**
The trajectory $\mathbf{x}(t)$ is a combination of these two effects. It's a vector that is simultaneously being **rotated** (due to the imaginary part $i\beta$) and having its **length scaled** (due to the real part $\alpha$). This combined motion is exactly what creates a spiral.

*   **Stable Spiral ($\alpha < 0$):** Rotation + Decaying Magnitude = Spiraling In.
*   **Unstable Spiral ($\alpha > 0$):** Rotation + Growing Magnitude = Spiraling Out.
*   **Center ($\alpha = 0$):** Rotation + Constant Magnitude = A perfect circle or ellipse.

## Summary

The behavior of a 2D linear dynamical system near its fixed point at the origin is completely characterized by the eigenvalues of its governing matrix $A$. By simply calculating the two eigenvalues, we can predict whether trajectories will converge, diverge, spiral, or orbit, providing a powerful tool for analyzing system stability without needing to solve the differential equations for every possible initial condition.

## Classification using Trace and Determinant

A more general way to classify the fixed point at the origin is to use the **trace** ($\tau = a+d$) and the **determinant** ($\Delta = ad-bc$) of the matrix $A = \begin{pmatrix} a & b \\ c & d \end{pmatrix}$.

The eigenvalues are the roots of the characteristic equation $\lambda^2 - \tau\lambda + \Delta = 0$, which are given by:

$$ \lambda_{1,2} = \frac{\tau \pm \sqrt{\tau^2 - 4\Delta}}{2} $$

The nature of the fixed point depends entirely on the values of $\tau$ and $\Delta$:
- The sign of $\Delta$ tells us if the eigenvalues have the same or opposite sign.
- The sign of $\tau$ tells us about the stability (since $\tau = \lambda_1 + \lambda_2$).
- The sign of the discriminant, $\tau^2 - 4\Delta$, tells us if the eigenvalues are real or complex.

This allows us to create a complete classification diagram in the $(\tau, \Delta)$-plane, as shown below. The parabola $\Delta = \tau^2/4$ is the boundary where eigenvalues switch from real (nodes) to complex (spirals).


In [ ]:
# Plotting the (Trace, Determinant)-plane for classification
fig, ax = plt.subplots(figsize=(10, 8))

tau = np.linspace(-4, 4, 500)

# Saddle points (Δ < 0)
ax.fill_between(tau, -4, 0, color="lightcoral", alpha=0.5)
ax.text(-2.5, -2, "Saddle Points", fontsize=14, ha="center")

# Stable Nodes (Δ > 0, τ < 0, τ²-4Δ > 0)
tau_stable = np.linspace(-4, 0, 250)
ax.fill_between(tau_stable, 0, tau_stable**2 / 4, color="dodgerblue", alpha=0.5)
ax.text(-2.5, 0.5, "Stable Nodes", fontsize=14, ha="center")

# Unstable Nodes (Δ > 0, τ > 0, τ²-4Δ > 0)
tau_unstable = np.linspace(0, 4, 250)
ax.fill_between(tau_unstable, 0, tau_unstable**2 / 4, color="limegreen", alpha=0.5)
ax.text(2.5, 0.5, "Unstable Nodes", fontsize=14, ha="center")


# Stable Spirals (Δ > 0, τ < 0, τ²-4Δ < 0)
ax.fill_between(tau_stable, tau_stable**2 / 4, 4, color="skyblue", alpha=0.5)
ax.text(-2, 3, "Stable Spirals", fontsize=14, ha="center")

# Unstable Spirals (Δ > 0, τ > 0, τ²-4Δ < 0)
ax.fill_between(tau_unstable, tau_unstable**2 / 4, 4, color="palegreen", alpha=0.5)
ax.text(2, 3, "Unstable Spirals", fontsize=14, ha="center")


# Plot the parabola Δ = τ²/4
ax.plot(
    tau, tau**2 / 4, "k-", lw=3, label="$\\tau^2 - 4\\Delta = 0$ (Repeated Eigenvalues)"
)

# Plot the axes
ax.axhline(0, color="black", lw=1)
ax.axvline(0, color="black", lw=1)

# Labels and Title
ax.set_xlabel("$\\tau$ (Trace of $A = a+d$)", fontsize=14)
ax.set_ylabel("$\\Delta$ (Determinant of $A=ad-bc$)", fontsize=14)
ax.set_title("Classification of 2D Linear Systems", fontsize=16)
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.legend()

# Add text for Center and Degenerate lines
ax.text(0.2, 2.5, "Centers", fontsize=14, rotation=90, va="center")
ax.text(2, -0.5, "Degenerate Nodes", fontsize=14, ha="center")

### Plotting from Trace and Determinant

We can now create a function that takes any pair of $(\tau, \Delta)$ from the diagram above and shows us what the corresponding phase portrait looks like. 
To do this, we can construct a special matrix called a **companion matrix**, which has a simple form but can represent any possible linear system:

$$ A = \begin{pmatrix} 0 & 1 \\ -\Delta & \tau \end{pmatrix} $$

This matrix has $\text{trace}(A) = \tau$ and $\det(A) = \Delta$, making it the perfect tool for exploring our classification map.

In [ ]:
def plot_portrait_from_trace_det(trace, determinant):
    """
    Constructs a companion matrix with a given trace and determinant,
    and plots its phase portrait.

    Args:
        trace (float): The desired trace (τ) of the system matrix.
        determinant (float): The desired determinant (Δ) of the system matrix.
    """
    # Companion matrix A = [[0, 1], [-Δ, τ]]
    A = np.array([[0, 1], [-determinant, trace]])

    title = f"Phase Portrait for τ={trace}, Δ={determinant}"
    plot_phase_portrait(A, title=title)

In [ ]:
# --- Example Usage ---
# Let's pick a point from the 'Stable Spiral' region of our diagram.
example_trace = -0.4
example_determinant = 2.0

# Check the condition: τ² - 4Δ = (-0.4)² - 4(2.0) = 0.16 - 8.0 = -7.84 < 0. It's a spiral!
# Since τ < 0, it's a stable spiral.
plot_portrait_from_trace_det(example_trace, example_determinant)

In [19]:
%matplotlib qt

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, fixed
import ipywidgets as widgets

# We assume 'plot_phase_portrait' is defined in a previous cell.
# This helper function constructs the matrix and calls the main plotter.
def plot_portrait_from_trace_det(trace, determinant):
    """
    Constructs a companion matrix with a given trace and determinant,
    and plots its phase portrait.
    """
    A = np.array([[0, 1], [-determinant, trace]])
    title = f"Phase Portrait for τ={trace}, Δ={determinant}"
    plot_phase_portrait(A, title=title)

# This is the function that will be controlled by the sliders.
def interactive_qt_plot(trace, determinant):
    """
    Closes any existing plot windows and creates a new one.
    This makes the Qt window interactive with the notebook sliders.
    """
    # Close all previously opened matplotlib figures. This is the key
    # to making the plot "refresh" in its separate window.
    plt.close('all')
    
    # The xkcd style needs to be active for each new plot.
    with plt.xkcd():
        plot_portrait_from_trace_det(trace, determinant)

# Use ipywidgets.interact to create the sliders and link them to our function.
# The semicolon at the end suppresses the function's output display in the notebook.
interact(interactive_qt_plot,
         trace=widgets.FloatSlider(min=-4.0, max=4.0, step=0.1, value=-0.4, description='Trace (τ)'),
         determinant=widgets.FloatSlider(min=-4.0, max=4.0, step=0.1, value=2.0, description='Determinant (Δ)'));


interactive(children=(FloatSlider(value=-0.4, description='Trace (τ)', max=4.0, min=-4.0), FloatSlider(value=2…